# Recommending Personalized Diet Categories and Foods Based on Nutrient Needs
**Author:** Qingyu (Wendy) Mao  
**EDA Notebook (PDF printout submission)**  
**Generated scaffold:** 2025-11-03 22:02

> This notebook is a scaffold you can run end-to-end. Fill in file paths in the **Configuration** cell, run all cells, and export to PDF.


## Table of Contents
- [1. Research Questions](#1-Research-Questions)
- [2. Motivation](#2-Motivation)
- [3. Data Setting](#3-Data-Setting)
- [4. Method (EDA-specific)](#4-Method-EDA-specific)
- [5. Configuration](#5-Configuration)
- [6. Helpers & Testing](#6-Helpers--Testing)
- [7. Load & Clean Datasets](#7-Load--Clean-Datasets)
- [8. EDA: Health/Diet Dataset](#8-EDA-HealthDiet-Dataset)
- [9. EDA: Nutrient Composition Dataset](#9-EDA-Nutrient-Composition-Dataset)
- [10. EDA: Join(s) & Relationship Between Datasets](#10-EDA-Joins--Relationship-Between-Datasets)
- [11. Result Validity: Hypotheses & Assumption Checks](#11-Result-Validity-Hypotheses--Assumption-Checks)
- [12. Challenge Goals & Updates](#12-Challenge-Goals--Updates)
- [13. Plan Evaluation](#13-Plan-Evaluation)
- [14. Trustworthiness: Tests, Assertions, Visuals](#14-Trustworthiness-Tests-Assertions-Visuals)
- [15. Appendix: Data Dictionaries](#15-Appendix-Data-Dictionaries)


## 1. Research Questions
1. **Group Differences:** Are there statistically significant differences in demographic and lifestyle features among different **diet recommendation groups**?
   - Planned tests: ANOVA (e.g., BMI by diet), t-tests (e.g., age for Low_Sodium vs others), chi-square (activity level vs diet).
2. **Prediction:** Can we predict the **best diet recommendation** for an individual from demographic/lifestyle features?
   - Planned: Compare kNN, Decision Tree, SVM (later stages).
3. **Food Lists:** What are the **top foods** that help individuals meet nutrient targets for each diet category?
   - Planned: Score foods by diet alignment once a diet label is predicted.
4. **Algorithm Selection:** Which ML algorithm performs best for personalized diet recommendations?
   - Planned: Systematic tuning & evaluation (later stages).


## 2. Motivation
Grounded in learnings from nutrition coursework, this project aims to translate population-level nutrient guidance into **personalized**, practical recommendations. The EDA here clarifies variable quality, missingness, plausible relationships, and the feasibility of robust modeling and food scoring.


## 3. Data Setting
**Datasets:**
- **Diet Recommendation Dataset** (Kaggle: patient-level demographics, lifestyle, indicators, and a recommended diet label). ~1000 rows × ~20 columns.
- **Nutrient Composition Dataset** (Kaggle: >300 foods with macronutrients and group classifications). ~300+ rows.

**Potential join keys:** food *group* / category mapping (for later recommendation stage). For EDA, we treat datasets independently and then explore simple joins by category names.


## 4. Method (EDA-specific)
- Inspect shapes, dtypes, and missingness.
- Summarize **variables of interest** (see Configuration), including seven-number summaries for quantitative variables and value counts for categorical variables.
- Create **at least two visualizations per dataset** with titles, axes labels, captions, and **alt text**.
- For multiple datasets, show basic relationships via joins or grouped comparisons.
- State **null hypotheses** and check assumptions (normality, equal variances, independence proxies) to prepare for statistical tests.
- Include doctests/assertions to ensure code correctness on small samples.


## 5. Configuration
Fill in your local file paths and map **column names** from your CSVs to the names used in this notebook.
> Tip: run the preview blocks after loading to verify your mappings.


In [ ]:

# === Configuration ===
HEALTH_CSV = "PATH/TO/diet_recommendations.csv"   # <-- update
FOOD_CSV   = "PATH/TO/nutrition.csv"              # <-- update

# Map your health dataset columns to the canonical names this notebook expects.
# Adjust keys on the left ONLY if you want different canonical names downstream.
HEALTH_COLS = {
    "age": "Age",
    "gender": "Gender",
    "height_cm": "Height",      # cm
    "weight_kg": "Weight",      # kg
    "bmi": "BMI",               # optional: will compute if missing
    "activity_level": "ActivityLevel",   # e.g., Low/Moderate/High
    "diet_label": "RecommendedDiet",     # target label
    # Optional useful columns you might have:
    "diet_type": "DietType",
    "has_hypertension": "Hypertension",
    "has_diabetes": "Diabetes"
}

# Variables of interest for EDA (health)
VARS_QUANT_HEALTH = ["Age", "Height", "Weight", "BMI"]
VARS_CAT_HEALTH   = ["Gender", "ActivityLevel", "RecommendedDiet"]

# Map your nutrient dataset columns to canonical names
FOOD_COLS = {
    "food_name": "Food",
    "food_group": "Group",
    "calories": "Calories",
    "protein_g": "Protein_g",
    "fat_g": "Fat_g",
    "carb_g": "Carb_g",
    # If dataset uses 't' for trace amounts, we'll coerce below
}

# Variables of interest for EDA (foods)
VARS_QUANT_FOOD = ["Calories", "Protein_g", "Fat_g", "Carb_g"]
VARS_CAT_FOOD   = ["Group"]


## 6. Helpers & Testing
Reusable functions with docstrings and small doctests for reliability.


In [ ]:

from __future__ import annotations
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np

def compute_bmi_series(weight_kg: pd.Series, height_cm: pd.Series) -> pd.Series:
    """
    Compute BMI from weight (kg) and height (cm).

    BMI = weight_kg / (height_m^2)

    >>> import pandas as pd
    >>> w = pd.Series([70, 60])
    >>> h = pd.Series([170, 160])
    >>> bmi = compute_bmi_series(w, h).round(2)
    >>> list(bmi)
    [24.22, 23.44]
    """
    height_m = height_cm / 100.0
    with np.errstate(divide="ignore", invalid="ignore"):
        bmi = weight_kg / (height_m ** 2)
    return bmi

def coerce_trace_to_float(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """
    Convert 't' (trace) to numeric small value (0.0 by default) for specified columns.

    >>> import pandas as pd
    >>> test = pd.DataFrame({"Protein_g": ["t", "0.3", "1.0"], "Fat_g": ["t", "t", "2"]})
    >>> out = coerce_trace_to_float(test, ["Protein_g", "Fat_g"])
    >>> out["Protein_g"].tolist(), out["Fat_g"].tolist()
    ([0.0, 0.3, 1.0], [0.0, 0.0, 2.0])
    """
    for c in cols:
        if c in df.columns:
            df[c] = df[c].replace("t", "0").astype(float)
    return df

def seven_number_summary(series: pd.Series) -> pd.Series:
    """
    Return mean, std, min, Q1, median, Q3, max for a numeric series.

    >>> import pandas as pd
    >>> s = pd.Series([1, 2, 3, 4, 5])
    >>> out = seven_number_summary(s)
    >>> round(out["mean"], 2), round(out["std"], 2), out["min"], out["max"]
    (3.0, 1.58, 1, 5)
    """
    desc = series.describe()  # count, mean, std, min, 25%, 50%, 75%, max
    return pd.Series({
        "mean": desc.get("mean"),
        "std": desc.get("std"),
        "min": desc.get("min"),
        "Q1": desc.get("25%"),
        "median": desc.get("50%"),
        "Q3": desc.get("75%"),
        "max": desc.get("max")
    })

def value_counts_frame(series: pd.Series) -> pd.DataFrame:
    """
    Return counts for categories as a 2-col DataFrame (value, count).

    >>> import pandas as pd
    >>> s = pd.Series(["A", "B", "A", "C", "B", "A"])
    >>> df = value_counts_frame(s)
    >>> list(df.iloc[0])
    ['A', 3]
    """
    vc = series.value_counts(dropna=False)
    return pd.DataFrame({"value": vc.index.astype(str), "count": vc.values})


In [ ]:

# Smoke tests
import doctest
_ = doctest.testmod(verbose=False)
print("Doctests passed.")


## 7. Load & Clean Datasets
Load, harmonize column names, and coerce types (including 'trace' handling for foods).

In [ ]:

import pandas as pd
import numpy as np

# Load
health_raw = pd.read_csv(HEALTH_CSV)
food_raw   = pd.read_csv(FOOD_CSV)

# Harmonize column names: rename based on provided maps if present
health = health_raw.rename(columns={v: k for k, v in HEALTH_COLS.items() if v in health_raw.columns})
food   = food_raw.rename(columns={v: k for k, v in FOOD_COLS.items() if v in food_raw.columns})

print("Health columns (after rename):", sorted(health.columns.tolist()))
print("Food columns (after rename):", sorted(food.columns.tolist()))

# Ensure expected columns exist (warn if not)
missing_health = [k for k in HEALTH_COLS.keys() if k not in health.columns]
missing_food   = [k for k in FOOD_COLS.keys() if k not in food.columns]
print("Missing (health):", missing_health)
print("Missing (food):", missing_food)

# Compute BMI if absent
if "bmi" not in health.columns and {"weight_kg","height_cm"}.issubset(health.columns):
    health["bmi"] = compute_bmi_series(health["weight_kg"], health["height_cm"])

# Coerce dtypes and clean foods
food = coerce_trace_to_float(food, ["protein_g","fat_g","carb_g"])

# Preview
display(health.head())
display(health.info())
display(food.head())
display(food.info())


## 8. EDA: Health/Diet Dataset
### 8.1 Size & Semantics
- **Rows:** individuals/patients
- **Columns:** demographics, lifestyle/health features, target diet label


In [ ]:

print("Health shape:", health.shape)
print("\nWhat rows represent: Each row is an individual (patient).")
print("What columns represent: Demographics, lifestyle/health features, and labels.")

# Missingness matrix (long format)
missing_health_df = health.isna().sum().reset_index()
missing_health_df.columns = ["column","n_missing"]
missing_health_df["pct_missing"] = (missing_health_df["n_missing"] / len(health)).round(4) * 100
missing_health_df = missing_health_df.sort_values("n_missing", ascending=False)
missing_health_df


**Interpretation (Missingness – Health):**
- Columns with nonzero missingness will be imputed/dropped based on proportion and relevance.
- Document plan: small numeric gaps → mean/median; categorical gaps → explicit 'Unknown' or mode.


### 8.2 Variables of Interest & Summaries
**Quantitative:** `Age`, `Height`, `Weight`, `BMI`

**Categorical:** `Gender`, `ActivityLevel`, `RecommendedDiet`


In [ ]:

# Quantitative summaries (seven-number)
quant_frames = {}
for col in VARS_QUANT_HEALTH:
    if col.lower() in health.columns:
        s = health[col.lower()]
        quant_frames[col] = seven_number_summary(s)

pd.DataFrame(quant_frames)


In [ ]:

# Categorical summaries (value counts)
cat_frames = {}
for col in VARS_CAT_HEALTH:
    if col.lower() in health.columns:
        s = health[col.lower()]
        cat_frames[col] = value_counts_frame(s)
        print(f"\n== {col} ==")
        display(cat_frames[col])


### 8.3 Visualizations (Health)
Create **at least two** plots. The examples below include a histogram and a grouped bar chart.

**Alt text template note:** Keep descriptions concrete (axes, ranges, trends).

In [ ]:

import matplotlib.pyplot as plt

# 1) BMI distribution by RecommendedDiet (histogram style via overlaid lines, binned data)
if {"bmi","recommendeddiet"}.issubset(health.columns):
    # Prepare binned counts per diet
    diets = health["recommendeddiet"].dropna().unique().tolist()
    bins = np.linspace(health["bmi"].min(), health["bmi"].max(), 30)

    plt.figure()
    for d in diets:
        s = health.loc[health["recommendeddiet"] == d, "bmi"].dropna()
        counts, edges = np.histogram(s, bins=bins)
        centers = 0.5*(edges[1:]+edges[:-1])
        plt.plot(centers, counts, label=str(d))
    plt.title("BMI Distribution by Recommended Diet")
    plt.xlabel("BMI")
    plt.ylabel("Count")
    plt.legend()
    plt.show()

# 2) Activity level vs Diet (grouped bar, as stacked counts per diet)
if {"activitylevel","recommendeddiet"}.issubset(health.columns):
    ct = pd.crosstab(health["activitylevel"], health["recommendeddiet"])
    ax = ct.plot(kind="bar", rot=0, legend=True, title="Counts by Activity Level and Recommended Diet")
    ax.set_xlabel("Activity Level")
    ax.set_ylabel("Count")
    plt.show()


**Caption (Plot 1):** BMI distributions vary across diet groups; peaks indicate common BMI ranges per diet label.  
**Alt text (Plot 1):** A line chart overlays binned BMI counts for several diet groups on the same axes, with BMI on the x-axis and counts on the y-axis; curves differ in height and shape across diets.

**Caption (Plot 2):** Activity levels show different count profiles across diet labels.  
**Alt text (Plot 2):** Grouped bars for activity levels on the x-axis show counts per recommended diet on the y-axis; bar heights vary by combination.

**Why these plots?** BMI is a core quantitative health metric related to diet recommendations; activity level is a key lifestyle factor potentially associated with diet labels.


## 9. EDA: Nutrient Composition Dataset
### 9.1 Size & Semantics
- **Rows:** foods
- **Columns:** nutrients (e.g., Calories, Protein_g, Fat_g, Carb_g), group/category


In [ ]:

print("Food shape:", food.shape)
print("\nWhat rows represent: Each row is a food item.")
print("What columns represent: Nutrient content and food grouping/category.")

# Missingness
missing_food_df = food.isna().sum().reset_index()
missing_food_df.columns = ["column","n_missing"]
missing_food_df["pct_missing"] = (missing_food_df["n_missing"] / len(food)).round(4) * 100
missing_food_df = missing_food_df.sort_values("n_missing", ascending=False)
missing_food_df


### 9.2 Variables of Interest & Summaries

In [ ]:

# Quant summaries
quant_frames_food = {}
for col in VARS_QUANT_FOOD:
    if col.lower() in food.columns:
        s = food[col.lower()]
        quant_frames_food[col] = seven_number_summary(s)

pd.DataFrame(quant_frames_food)


In [ ]:

# Categorical
for col in VARS_CAT_FOOD:
    if col.lower() in food.columns:
        print(f"\n== {col} ==")
        display(value_counts_frame(food[col.lower()]))


### 9.3 Visualizations (Foods)
Examples: nutrients by group, pairwise relationships.


In [ ]:

# 1) Calories by Group (box-like view via summary lines/points due to no seaborn rule)
import matplotlib.pyplot as plt
if {"group","calories"}.issubset(food.columns):
    groups = food["group"].dropna().unique().tolist()
    data = [food.loc[food["group"]==g, "calories"].dropna().values for g in groups]

    # Simple custom box-ish: show median and IQR as lines
    plt.figure(figsize=(10,5))
    for i, arr in enumerate(data):
        if len(arr)==0:
            continue
        q1, med, q3 = np.percentile(arr, [25,50,75])
        plt.plot([i, i], [q1, q3])
        plt.plot(i, med, marker="o")
    plt.xticks(range(len(groups)), groups, rotation=45, ha="right")
    plt.title("Calories by Food Group (median with IQR whiskers)")
    plt.xlabel("Food Group")
    plt.ylabel("Calories")
    plt.tight_layout()
    plt.show()

# 2) Protein vs Calories scatter
if {"protein_g","calories"}.issubset(food.columns):
    plt.figure()
    plt.scatter(food["calories"], food["protein_g"])
    plt.title("Protein vs Calories")
    plt.xlabel("Calories")
    plt.ylabel("Protein (g)")
    plt.show()


**Caption (Plot 1):** Median calories and spread differ by food group.  
**Alt text (Plot 1):** For each food group on the x-axis, a vertical line shows the interquartile range of calories with a point marking the median on the y-axis.  
**Caption (Plot 2):** Higher-calorie foods are not always high in protein; the scatter suggests varying protein-density.  
**Alt text (Plot 2):** A scatterplot with calories on the x-axis and protein grams on the y-axis shows points widely dispersed with no single linear trend.  
**Why these plots?** They reveal nutrient distribution differences across groups and a basic density relationship for protein.


## 10. EDA: Joins & Relationship Between Datasets
Since diet recommendations will later be translated into **food suggestions**, here we preview one simple bridge: comparing nutrient summaries by **Group** and noting which groups might align with diet labels (e.g., Low_Sodium, Low_Carb, Balanced). In the full project, a more nuanced scoring will be implemented.


In [ ]:

# Example: aggregate nutrients by food Group
if "group" in food.columns:
    agg = food.groupby("group", dropna=True)[["calories","protein_g","fat_g","carb_g"]].mean().round(2)
    agg = agg.sort_values("calories")
    display(agg)


## 11. Result Validity: Hypotheses & Assumption Checks
**Null Hypotheses**
- **ANOVA (BMI by diet)**: μ_Balanced = μ_Low_Carb = μ_Low_Sodium (and any other diets).  
- **t-test (Age: Low_Sodium vs others)**: μ_Low_Sodium = μ_Others.  
- **Chi-square (Activity vs Diet)**: ActivityLevel ⟂ RecommendedDiet (independence).

**Assumptions:**
- ANOVA/t-test: (approx.) normality within groups; homogeneity of variances; independent observations.  
- Chi-square: expected cell counts not too small; independent observations.


In [ ]:

from scipy import stats
import numpy as np
import pandas as pd

# ANOVA assumption checks (Shapiro for each group; Levene for var homogeneity)
if {"bmi","recommendeddiet"}.issubset(health.columns):
    groups = [g.dropna() for _, g in health.groupby("recommendeddiet")["bmi"]]
    # Normality (informal; large N makes it robust)
    shapiro_results = [stats.shapiro(g.sample(min(len(g), 500), random_state=0)) if len(g)>=3 else None for g in groups]
    # Homogeneity
    if all(len(g) > 1 for g in groups):
        levene_stat, levene_p = stats.levene(*groups, center="median")
        print("Levene test (variances equal?): stat=%.3f, p=%.4f" % (levene_stat, levene_p))
    else:
        print("Levene skipped (insufficient group sizes).")

# Chi-square expected counts check
if {"activitylevel","recommendeddiet"}.issubset(health.columns):
    ct = pd.crosstab(health["activitylevel"], health["recommendeddiet"])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    print("Chi-square test: chi2=%.3f, p=%.4f, dof=%d" % (chi2, p, dof))
    # Inspect small expected counts
    small_cells = (expected < 5).sum()
    print("Cells with expected < 5:", small_cells)


## 12. Challenge Goals & Updates
- **Statistical Hypothesis Testing:** The checks above inform feasibility; report any violations and remedies (transformations, robust tests, nonparametrics).
- **Advanced ML:** Deferred to modeling phase; EDA findings will guide feature encoding and target balance considerations.


## 13. Plan Evaluation
Reflect on time spent so far vs. estimates; list remaining tasks (feature engineering, modeling, evaluation, food scoring) with updated time estimates.


## 14. Trustworthiness: Tests, Assertions, Visuals
- This notebook includes doctests for helpers and sanity-check prints.  
- Add assertions after cleaning (e.g., BMI nonnegative, expected columns present).  
- Use small synthetic frames in new cells to test edge cases (e.g., all 't' values, missing categories).


## 15. Appendix: Data Dictionaries
Document meanings and units for each **variable of interest** after you load your specific datasets.
